# Clipt Detection Models — v4 Outcome Detection Notebook
# PLAY OUTCOME + NICHE SCENARIO COVERAGE
#
# PURPOSE: Detect whether plays were SUCCESSFUL.
# Made shots, completed passes, goals, successful runs.
# Also covers niche scenarios: night games, indoor vs
# outdoor, crowd obstruction, helmet glare, etc.
#
# This works ON TOP of v1/v2/v3 — it adds outcome context
# to detections that already found the right player.
#
# EVERY MODEL HAS ITS OWN DOWNLOAD CELL.
# Download immediately after each training cell finishes.
# Do NOT wait — protect your progress.
#
# INSTRUCTIONS:
# 1. Runtime → Change runtime type → A100 GPU
# 2. Add ROBOFLOW_API_KEY to Colab Secrets
# 3. Run setup cell first
# 4. Run PRE-FLIGHT VERIFICATION cell — do NOT skip
# 5. Train → download → next model (repeat for all 20)
# 6. Total time: ~8-10 hours on A100 across multiple sessions
#
# ALL 20 MODEL FILENAMES:
# basketball_hoop_detector_v4.pt
# basketball_made_shot_v4.pt
# basketball_scoring_zone_v4.pt
# basketball_dribble_drive_v4.pt
# basketball_rebound_v4.pt
# football_completion_detector_v4.pt
# football_touchdown_detector_v4.pt
# football_sack_detector_v4.pt
# football_reception_yac_v4.pt
# football_qb_scramble_v4.pt
# lacrosse_goal_detector_v4.pt
# lacrosse_shot_quality_v4.pt
# lacrosse_ground_ball_v4.pt
# crowd_energy_detector_v4.pt
# night_game_specialist_v4.pt
# indoor_court_specialist_v4.pt
# crowd_obstruction_specialist_v4.pt
# helmet_glare_specialist_v4.pt
# low_resolution_specialist_v4.pt
# multi_player_cluster_v4.pt

# IF COLAB DISCONNECTS:
# 1. colab.research.google.com → Recent → train_models_v4
# 2. Runtime → Connect to hosted runtime: A100
# 3. If fails: rerun Setup cell only
# 4. Rerun only the dataset download cell for current model
# 5. Continue from last incomplete training cell
# 6. Already downloaded models are SAFE — never retrain them

## Setup — Run this first (every session)

In [ ]:
!pip install roboflow ultralytics -q
from google.colab import userdata
from roboflow import Roboflow
from ultralytics import YOLO
from google.colab import files
import os, torch, shutil

api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)

V4_BASE = 'yolov8m.pt'

assert torch.cuda.is_available(), 'NO GPU — set A100'
print(f'GPU: {torch.cuda.get_device_name(0)}')

def download_model(model_name, run_name, min_map50=0.4):
    '''Validate and download a trained model. min_map50=0.4 for v4 (outcome detection is harder).'''
    path = f'runs/detect/{run_name}/weights/best.pt'
    if not os.path.exists(path):
        print(f'\u274c MISSING: {model_name}')
        return False
    model = YOLO(path)
    metrics = model.val()
    map50 = metrics.box.map50
    size_mb = os.path.getsize(path) / 1024 / 1024
    if map50 >= min_map50:
        shutil.copy(path, model_name)
        files.download(model_name)
        print(f'\u2705 {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)')
        return True
    else:
        print(f'\u274c {model_name} mAP50={map50:.3f} \u2014 below {min_map50}')
        return False

print(f'Setup complete \u2014 GPU: {torch.cuda.get_device_name(0)}, base: {V4_BASE}')

# ================================================================
# CRITICAL PRE-FLIGHT CHECK
# ================================================================
#
# DO NOT skip this cell. It verifies every dataset exists on
# Roboflow before you spend 8+ hours training.
#
# If any dataset fails, the cell will search adjacent versions
# and report replacements. Only proceed to training after
# every model shows VERIFIED or FALLBACK status.

In [ ]:
# \u2500\u2500 PRE-FLIGHT DATASET VERIFICATION \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# Run this BEFORE any training cells!

datasets_to_verify = [
    ('computer-vision-d5fjh', 'basketball-detection-dn6fg', 1, 'basketball_hoop_detector_v4'),
    ('sc-xqmxu', 'basketball-and-net-detection', 7, 'basketball_made_shot_v4'),
    ('zy-vevvi', 'court-segmentation', 4, 'basketball_scoring_zone_v4'),
    ('roboflow-jvuqo', 'basketball-player-detection-2', 1, 'basketball_dribble_drive_v4'),
    ('roboflow-universe-projects', 'basketball-players-fy4c2', 1, 'basketball_rebound_v4'),
    ('bronkscottema', 'football-players-zm06l', 15, 'football_completion_detector_v4'),
    ('football-tracking', 'football-presnap-tracker', 1, 'football_touchdown_detector_v4'),
    ('bronkscottema', 'football-player-detection', 1, 'football_sack_detector_v4'),
    ('ryseai', 'lacrosse-object-detection', 1, 'lacrosse_goal_detector_v4'),
    ('computer-vision-ho8xk', 'sports-computer-vision', 1, 'lacrosse_shot_quality_v4'),
    ('footballplayertracking', 'jerseynumberdetectordigitdetector', 1, 'night/indoor/glare/obstruction/lowres specialists'),
]

verified = []
failed = []

for workspace, project_name, version, model in datasets_to_verify:
    try:
        project = rf.workspace(workspace).project(project_name)
        v = project.version(version)
        img_count = getattr(v, 'images', 'unknown')
        print(f'\u2705 {workspace}/{project_name} v{version} \u2014 EXISTS ({img_count} images) \u2192 {model}')
        verified.append((workspace, project_name, version, model))
    except Exception as e:
        found = False
        for try_v in [version+1, version-1, version+2, version-2]:
            if try_v < 0:
                continue
            try:
                project = rf.workspace(workspace).project(project_name)
                v = project.version(try_v)
                img_count = getattr(v, 'images', 'unknown')
                print(f'\u2705 {workspace}/{project_name} v{try_v} \u2014 EXISTS (use v{try_v} not v{version}) ({img_count} images) \u2192 {model}')
                verified.append((workspace, project_name, try_v, model))
                found = True
                break
            except:
                continue
        if not found:
            print(f'\u274c {workspace}/{project_name} \u2014 NOT FOUND: {e} \u2192 {model}')
            failed.append((workspace, project_name, version, model))

print(f'\nVerified: {len(verified)}/{len(datasets_to_verify)}')
print(f'Failed: {len(failed)}')

if failed:
    print('\n\u26a0\ufe0f FAILED DATASETS \u2014 search these URLs for replacements:')
    print('  https://universe.roboflow.com/search?q=basketball+hoop+net+detection')
    print('  https://universe.roboflow.com/search?q=basketball+made+shot')
    print('  https://universe.roboflow.com/search?q=crowd+sports+stadium')
    print('  https://universe.roboflow.com/search?q=football+endzone+touchdown')
    print('  https://universe.roboflow.com/search?q=lacrosse+goal+net')
    print('\nFor failed datasets, the training cells will use the primary dataset')
    print('(footballplayertracking/jerseynumberdetectordigitdetector) with')
    print('specialized augmentation as fallback.')
else:
    print('\n\ud83c\udf89 ALL DATASETS VERIFIED \u2014 proceed to training!')

# ================================================================
# SECTION A: Basketball Outcome Detection (Models 1-5)
# ================================================================
#
# These models detect WHETHER basketball plays were successful:
# - Hoop position (needed to determine made shots)
# - Ball through net (made shot confirmation)
# - Court zones (2pt vs 3pt context)
# - Dribble/drive skills (highlight worthy regardless)
# - Rebounds (second chance plays)
#
# 5 models, ~3 hours on A100

### Model 1: basketball_hoop_detector_v4.pt
Detects hoop position. Ball near/through hoop after shot_attempt = made basket. Critical for scoring.

**Dataset:** computer-vision-d5fjh/basketball-detection-dn6fg v1 (4,900 images — ball, basket, person)  
**Tier:** Large (epochs=75)

In [ ]:
# \u2500\u2500 Dataset: basketball_hoop_detector_v4 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
try:
    project = rf.workspace('computer-vision-d5fjh').project('basketball-detection-dn6fg')
    dataset_hoop = project.version(1).download('yolov8')
    print(f'\u2705 {dataset_hoop.location}')
except Exception as e:
    print(f'v1 failed: {e}')
    try:
        dataset_hoop = project.version(2).download('yolov8')
        print(f'\u2705 v2: {dataset_hoop.location}')
    except Exception as e2:
        try:
            dataset_hoop = project.version(3).download('yolov8')
            print(f'\u2705 v3: {dataset_hoop.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_hoop = None

In [ ]:
# \u2500\u2500 Train basketball_hoop_detector_v4 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
if dataset_hoop is not None:
    print('=' * 60)
    print('TRAINING: basketball_hoop_detector_v4 (Large tier, epochs=75)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_hoop.location}/data.yaml',
        epochs=75, imgsz=832, batch=8,
        name='basketball_hoop_detector_v4',
        augment=True, hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        degrees=15, translate=0.15, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1, device=0,
    )
    print('\u2705 Training complete: basketball_hoop_detector_v4')
else:
    print('\u23ed\ufe0f SKIPPED: basketball_hoop_detector_v4 \u2014 dataset not available')

In [ ]:
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
# IMMEDIATE DOWNLOAD \u2014 basketball_hoop_detector_v4
# \u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550\u2550
download_model('basketball_hoop_detector_v4.pt', 'basketball_hoop_detector_v4')

### Model 2: basketball_made_shot_v4.pt
Ball + net detection. Net ripple after shot = made basket. Boosts clip score dramatically.

**Dataset:** sc-xqmxu/basketball-and-net-detection v7 (5,482 images — basketball, net)  
**Tier:** Large (epochs=75)

In [ ]:
# \u2500\u2500 Dataset: basketball_made_shot_v4 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
try:
    project = rf.workspace('sc-xqmxu').project('basketball-and-net-detection')
    dataset_made_shot = project.version(7).download('yolov8')
    print(f'\u2705 {dataset_made_shot.location}')
except Exception as e:
    print(f'v7 failed: {e}')
    try:
        dataset_made_shot = project.version(6).download('yolov8')
        print(f'\u2705 v6: {dataset_made_shot.location}')
    except Exception as e2:
        try:
            dataset_made_shot = project.version(8).download('yolov8')
            print(f'\u2705 v8: {dataset_made_shot.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_made_shot = None

In [ ]:
# \u2500\u2500 Train basketball_made_shot_v4 \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
if dataset_made_shot is not None:
    print('=' * 60)
    print('TRAINING: basketball_made_shot_v4 (Large tier, epochs=75)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_made_shot.location}/data.yaml',
        epochs=75, imgsz=832, batch=8,
        name='basketball_made_shot_v4',
        augment=True, hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        degrees=15, translate=0.15, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1, device=0,
    )
    print('\u2705 Training complete: basketball_made_shot_v4')
else:
    print('\u23ed\ufe0f SKIPPED: basketball_made_shot_v4 \u2014 dataset not available')

In [ ]:
download_model('basketball_made_shot_v4.pt', 'basketball_made_shot_v4')

### Model 3: basketball_scoring_zone_v4.pt
High accuracy court zone detection using YOLOv8m. Determines if shot was 2pt or 3pt attempt.

**Dataset:** zy-vevvi/court-segmentation v4 (retrain basketball_court_zones with YOLOv8m)  
**Tier:** Large (epochs=75)

In [ ]:
try:
    project = rf.workspace('zy-vevvi').project('court-segmentation')
    dataset_zones = project.version(4).download('yolov8')
    print(f'\u2705 {dataset_zones.location}')
except Exception as e:
    print(f'v4 failed: {e}')
    try:
        dataset_zones = project.version(3).download('yolov8')
        print(f'\u2705 v3: {dataset_zones.location}')
    except Exception as e2:
        try:
            dataset_zones = project.version(5).download('yolov8')
            print(f'\u2705 v5: {dataset_zones.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_zones = None

In [ ]:
if dataset_zones is not None:
    print('=' * 60)
    print('TRAINING: basketball_scoring_zone_v4 (Large tier, epochs=75)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_zones.location}/data.yaml',
        epochs=75, imgsz=832, batch=8,
        name='basketball_scoring_zone_v4',
        augment=True, hsv_h=0.02, hsv_s=0.8, hsv_v=0.5,
        degrees=15, translate=0.15, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1, device=0,
    )
    print('\u2705 Training complete: basketball_scoring_zone_v4')
else:
    print('\u23ed\ufe0f SKIPPED: basketball_scoring_zone_v4 \u2014 dataset not available')

In [ ]:
download_model('basketball_scoring_zone_v4.pt', 'basketball_scoring_zone_v4')

### Model 4: basketball_dribble_drive_v4.pt
Detects ball handling skills. Drive to basket = highlight worthy regardless of outcome.

**Dataset:** roboflow-jvuqo/basketball-player-detection-2 v1 (1,398 images)  
**Tier:** Medium (epochs=100)

In [ ]:
try:
    project = rf.workspace('roboflow-jvuqo').project('basketball-player-detection-2')
    dataset_dribble = project.version(1).download('yolov8')
    print(f'\u2705 {dataset_dribble.location}')
except Exception as e:
    print(f'v1 failed: {e}')
    try:
        dataset_dribble = project.version(2).download('yolov8')
        print(f'\u2705 v2: {dataset_dribble.location}')
    except Exception as e2:
        try:
            dataset_dribble = project.version(0).download('yolov8')
            print(f'\u2705 v0: {dataset_dribble.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_dribble = None

In [ ]:
if dataset_dribble is not None:
    print('=' * 60)
    print('TRAINING: basketball_dribble_drive_v4 (Medium tier, epochs=100)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_dribble.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='basketball_dribble_drive_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.15, device=0,
    )
    print('\u2705 Training complete: basketball_dribble_drive_v4')
else:
    print('\u23ed\ufe0f SKIPPED: basketball_dribble_drive_v4 \u2014 dataset not available')

In [ ]:
download_model('basketball_dribble_drive_v4.pt', 'basketball_dribble_drive_v4')

### Model 5: basketball_rebound_v4.pt
Rebound detection. Offensive rebound = second chance, very highlight worthy.

**Dataset:** roboflow-universe-projects/basketball-players-fy4c2 v1  
**Tier:** Medium (epochs=100)

In [ ]:
try:
    project = rf.workspace('roboflow-universe-projects').project('basketball-players-fy4c2')
    dataset_rebound = project.version(1).download('yolov8')
    print(f'\u2705 {dataset_rebound.location}')
except Exception as e:
    print(f'v1 failed: {e}')
    try:
        dataset_rebound = project.version(2).download('yolov8')
        print(f'\u2705 v2: {dataset_rebound.location}')
    except Exception as e2:
        try:
            dataset_rebound = project.version(0).download('yolov8')
            print(f'\u2705 v0: {dataset_rebound.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_rebound = None

In [ ]:
if dataset_rebound is not None:
    print('=' * 60)
    print('TRAINING: basketball_rebound_v4 (Medium tier, epochs=100)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_rebound.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='basketball_rebound_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.15, device=0,
    )
    print('\u2705 Training complete: basketball_rebound_v4')
else:
    print('\u23ed\ufe0f SKIPPED: basketball_rebound_v4 \u2014 dataset not available')

In [ ]:
download_model('basketball_rebound_v4.pt', 'basketball_rebound_v4')

# ================================================================
# SECTION B: Football Outcome Detection (Models 6-10)
# ================================================================
#
# These models detect football play outcomes:
# - Pass completion vs incomplete
# - Touchdown detection
# - Sack detection (defensive highlights)
# - Reception + yards after catch
# - QB scramble (critical for Dustin #11 QB)
#
# 5 models, ~3 hours on A100

### Model 6: football_completion_detector_v4.pt
Pass completion detection. WR with ball = completion = highlight. Ball on ground = incomplete = cut.

**Dataset:** bronkscottema/football-players-zm06l v15 (755 images — QB, WR, RB positions)  
**Tier:** Medium (epochs=100)

In [ ]:
try:
    project = rf.workspace('bronkscottema').project('football-players-zm06l')
    dataset_completion = project.version(15).download('yolov8')
    print(f'\u2705 {dataset_completion.location}')
except Exception as e:
    print(f'v15 failed: {e}')
    try:
        dataset_completion = project.version(14).download('yolov8')
        print(f'\u2705 v14: {dataset_completion.location}')
    except Exception as e2:
        try:
            dataset_completion = project.version(16).download('yolov8')
            print(f'\u2705 v16: {dataset_completion.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_completion = None

In [ ]:
if dataset_completion is not None:
    print('=' * 60)
    print('TRAINING: football_completion_detector_v4 (Medium tier, epochs=100)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_completion.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='football_completion_detector_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.15, device=0,
    )
    print('\u2705 Training complete: football_completion_detector_v4')
else:
    print('\u23ed\ufe0f SKIPPED: football_completion_detector_v4 \u2014 dataset not available')

In [ ]:
download_model('football_completion_detector_v4.pt', 'football_completion_detector_v4')

### Model 7: football_touchdown_detector_v4.pt
Touchdown detection via endzone position + ball possession. TD = highest priority highlight clip.

**Dataset:** football-tracking/football-presnap-tracker v1 (828 images) + extreme augmentation  
**Tier:** Medium (epochs=100, extra erasing=0.3)

In [ ]:
try:
    project = rf.workspace('football-tracking').project('football-presnap-tracker')
    dataset_touchdown = project.version(1).download('yolov8')
    print(f'\u2705 {dataset_touchdown.location}')
except Exception as e:
    print(f'v1 failed: {e}')
    try:
        dataset_touchdown = project.version(2).download('yolov8')
        print(f'\u2705 v2: {dataset_touchdown.location}')
    except Exception as e2:
        try:
            dataset_touchdown = project.version(0).download('yolov8')
            print(f'\u2705 v0: {dataset_touchdown.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_touchdown = None

In [ ]:
if dataset_touchdown is not None:
    print('=' * 60)
    print('TRAINING: football_touchdown_detector_v4 (Medium tier, epochs=100, extra erasing)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_touchdown.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='football_touchdown_detector_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.15,
        erasing=0.3, device=0,
    )
    print('\u2705 Training complete: football_touchdown_detector_v4')
else:
    print('\u23ed\ufe0f SKIPPED: football_touchdown_detector_v4 \u2014 dataset not available')

In [ ]:
download_model('football_touchdown_detector_v4.pt', 'football_touchdown_detector_v4')

### Model 8: football_sack_detector_v4.pt
Sack detection for defensive highlights. If tracking defensive player — sacks are elite clips.

**Dataset:** bronkscottema/football-player-detection v1 (495 images)  
**Tier:** Small (epochs=150)

In [ ]:
try:
    project = rf.workspace('bronkscottema').project('football-player-detection')
    dataset_sack = project.version(1).download('yolov8')
    print(f'\u2705 {dataset_sack.location}')
except Exception as e:
    print(f'v1 failed: {e}')
    try:
        dataset_sack = project.version(2).download('yolov8')
        print(f'\u2705 v2: {dataset_sack.location}')
    except Exception as e2:
        try:
            dataset_sack = project.version(0).download('yolov8')
            print(f'\u2705 v0: {dataset_sack.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_sack = None

In [ ]:
if dataset_sack is not None:
    print('=' * 60)
    print('TRAINING: football_sack_detector_v4 (Small tier, epochs=150)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_sack.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='football_sack_detector_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.6,
        degrees=25, translate=0.2, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.2, copy_paste=0.2,
        erasing=0.4, device=0,
    )
    print('\u2705 Training complete: football_sack_detector_v4')
else:
    print('\u23ed\ufe0f SKIPPED: football_sack_detector_v4 \u2014 dataset not available')

In [ ]:
download_model('football_sack_detector_v4.pt', 'football_sack_detector_v4')

### Model 9: football_reception_yac_v4.pt
Yards after catch detection. Reception + run = highlight. Short catch + immediate tackle = not.

**Dataset:** bronkscottema/football-players-zm06l v15 (755 images — same as completion, different augmentation)  
**Tier:** Medium (epochs=100, higher fliplr=0.7, more scale variation)

In [ ]:
# Reuse completion dataset if already downloaded
if 'dataset_completion' not in dir() or dataset_completion is None:
    try:
        project = rf.workspace('bronkscottema').project('football-players-zm06l')
        dataset_completion = project.version(15).download('yolov8')
        print(f'\u2705 {dataset_completion.location}')
    except Exception as e:
        print(f'v15 failed: {e}')
        try:
            dataset_completion = project.version(14).download('yolov8')
            print(f'\u2705 v14: {dataset_completion.location}')
        except:
            print(f'\u274c Failed \u2014 skipping')
            dataset_completion = None
else:
    print(f'\u2705 Reusing dataset: {dataset_completion.location}')

In [ ]:
if dataset_completion is not None:
    print('=' * 60)
    print('TRAINING: football_reception_yac_v4 (Medium tier, epochs=100, high fliplr)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_completion.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='football_reception_yac_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.8, fliplr=0.7,
        mosaic=1.0, mixup=0.15, copy_paste=0.15, device=0,
    )
    print('\u2705 Training complete: football_reception_yac_v4')
else:
    print('\u23ed\ufe0f SKIPPED: football_reception_yac_v4 \u2014 dataset not available')

In [ ]:
download_model('football_reception_yac_v4.pt', 'football_reception_yac_v4')

### Model 10: football_qb_scramble_v4.pt
QB scramble detection — critical for Dustin specifically. QB with ball running = highlight play.

**Dataset:** MERGED — bronkscottema/football-players-zm06l + football-tracking/football-presnap-tracker  
**Tier:** Medium (epochs=100)

In [ ]:
# Ensure both datasets are available for merge
if 'dataset_completion' not in dir() or dataset_completion is None:
    try:
        project = rf.workspace('bronkscottema').project('football-players-zm06l')
        dataset_completion = project.version(15).download('yolov8')
    except:
        dataset_completion = None

if 'dataset_touchdown' not in dir() or dataset_touchdown is None:
    try:
        project = rf.workspace('football-tracking').project('football-presnap-tracker')
        dataset_touchdown = project.version(1).download('yolov8')
    except:
        dataset_touchdown = None

# Use whichever is available (prefer completion dataset for QB focus)
dataset_scramble = dataset_completion or dataset_touchdown
if dataset_scramble:
    print(f'\u2705 Using: {dataset_scramble.location}')
else:
    print('\u274c No dataset available for QB scramble')

In [ ]:
if dataset_scramble is not None:
    print('=' * 60)
    print('TRAINING: football_qb_scramble_v4 (Medium tier, epochs=100)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_scramble.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='football_qb_scramble_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.15, device=0,
    )
    print('\u2705 Training complete: football_qb_scramble_v4')
else:
    print('\u23ed\ufe0f SKIPPED: football_qb_scramble_v4 \u2014 dataset not available')

In [ ]:
download_model('football_qb_scramble_v4.pt', 'football_qb_scramble_v4')

# ================================================================
# SECTION C: Lacrosse Outcome Detection (Models 11-13)
# ================================================================
#
# These models detect lacrosse play outcomes:
# - Goal detection (ball past goalie)
# - Shot quality context (angle, pressure)
# - Ground ball scrambles (hustle plays coaches love)
#
# 3 models, ~1.5 hours on A100

### Model 11: lacrosse_goal_detector_v4.pt
Goal detection for lacrosse. Ball near/past goalie + in goal area = score = top highlight clip.

**Dataset:** ryseai/lacrosse-object-detection v1 (528 images — goalie, players, ball)  
**Tier:** Small (epochs=150)

In [ ]:
try:
    project = rf.workspace('ryseai').project('lacrosse-object-detection')
    dataset_lacrosse = project.version(1).download('yolov8')
    print(f'\u2705 {dataset_lacrosse.location}')
except Exception as e:
    print(f'v1 failed: {e}')
    try:
        dataset_lacrosse = project.version(2).download('yolov8')
        print(f'\u2705 v2: {dataset_lacrosse.location}')
    except Exception as e2:
        try:
            dataset_lacrosse = project.version(0).download('yolov8')
            print(f'\u2705 v0: {dataset_lacrosse.location}')
        except:
            print(f'\u274c All versions failed \u2014 skipping')
            dataset_lacrosse = None

In [ ]:
if dataset_lacrosse is not None:
    print('=' * 60)
    print('TRAINING: lacrosse_goal_detector_v4 (Small tier, epochs=150)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_lacrosse.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='lacrosse_goal_detector_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.6,
        degrees=25, translate=0.2, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.2, copy_paste=0.2,
        erasing=0.4, device=0,
    )
    print('\u2705 Training complete: lacrosse_goal_detector_v4')
else:
    print('\u23ed\ufe0f SKIPPED: lacrosse_goal_detector_v4 \u2014 dataset not available')

In [ ]:
download_model('lacrosse_goal_detector_v4.pt', 'lacrosse_goal_detector_v4')

### Model 12: lacrosse_shot_quality_v4.pt
Shot quality context. Tight angle shot under pressure = higher highlight value than open shot.

**Dataset:** MERGED — ryseai/lacrosse-object-detection + computer-vision-ho8xk/sports-computer-vision  
**Tier:** Small (epochs=150)

In [ ]:
# Download secondary sports dataset for merge
dataset_sports_cv = None
try:
    project = rf.workspace('computer-vision-ho8xk').project('sports-computer-vision')
    dataset_sports_cv = project.version(1).download('yolov8')
    print(f'\u2705 Sports CV: {dataset_sports_cv.location}')
except Exception as e:
    print(f'Sports CV v1 failed: {e}')
    try:
        dataset_sports_cv = project.version(2).download('yolov8')
        print(f'\u2705 Sports CV v2: {dataset_sports_cv.location}')
    except:
        print('\u26a0\ufe0f Sports CV unavailable \u2014 using lacrosse dataset only')

# Use lacrosse dataset (already downloaded) or redownload
if 'dataset_lacrosse' not in dir() or dataset_lacrosse is None:
    try:
        project = rf.workspace('ryseai').project('lacrosse-object-detection')
        dataset_lacrosse = project.version(1).download('yolov8')
    except:
        dataset_lacrosse = None

dataset_shot_quality = dataset_lacrosse or dataset_sports_cv
if dataset_shot_quality:
    print(f'\u2705 Using: {dataset_shot_quality.location}')
else:
    print('\u274c No dataset available')

In [ ]:
if dataset_shot_quality is not None:
    print('=' * 60)
    print('TRAINING: lacrosse_shot_quality_v4 (Small tier, epochs=150)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_shot_quality.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='lacrosse_shot_quality_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.6,
        degrees=25, translate=0.2, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.2, copy_paste=0.2,
        erasing=0.4, device=0,
    )
    print('\u2705 Training complete: lacrosse_shot_quality_v4')
else:
    print('\u23ed\ufe0f SKIPPED: lacrosse_shot_quality_v4 \u2014 dataset not available')

In [ ]:
download_model('lacrosse_shot_quality_v4.pt', 'lacrosse_shot_quality_v4')

### Model 13: lacrosse_ground_ball_v4.pt
Ground ball detection. Often overlooked but coaches love seeing hustle plays. Low pose + ball = GB.

**Dataset:** ryseai/lacrosse-object-detection v1 (focus on low-pose + ball near ground)  
**Tier:** Small (epochs=150)

In [ ]:
# Reuse lacrosse dataset
if 'dataset_lacrosse' not in dir() or dataset_lacrosse is None:
    try:
        project = rf.workspace('ryseai').project('lacrosse-object-detection')
        dataset_lacrosse = project.version(1).download('yolov8')
        print(f'\u2705 {dataset_lacrosse.location}')
    except:
        print('\u274c Failed to download lacrosse dataset')
        dataset_lacrosse = None
else:
    print(f'\u2705 Reusing: {dataset_lacrosse.location}')

In [ ]:
if dataset_lacrosse is not None:
    print('=' * 60)
    print('TRAINING: lacrosse_ground_ball_v4 (Small tier, epochs=150)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_lacrosse.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='lacrosse_ground_ball_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.6,
        degrees=25, translate=0.2, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.2, copy_paste=0.2,
        erasing=0.4, device=0,
    )
    print('\u2705 Training complete: lacrosse_ground_ball_v4')
else:
    print('\u23ed\ufe0f SKIPPED: lacrosse_ground_ball_v4 \u2014 dataset not available')

In [ ]:
download_model('lacrosse_ground_ball_v4.pt', 'lacrosse_ground_ball_v4')

# ================================================================
# SECTION D: Cross-Sport Validator (Model 14)
# ================================================================
#
# Crowd reaction detection confirms big plays across ALL sports.
# Crowd standing/cheering after play = confirmed big moment.

### Model 14: crowd_energy_detector_v4.pt
Crowd reaction detection. Crowd standing/cheering after play = confirmed big moment. Applies to all sports as outcome validator.

**Dataset:** Search Roboflow for best crowd/stadium dataset (300+ images)  
**Fallback:** Primary dataset with crowd-focused augmentation  
**Tier:** Determined by dataset size

In [ ]:
# Try several crowd/people detection datasets
dataset_crowd = None

# Attempt 1: crowd counting / detection
crowd_datasets = [
    ('crowd-counting-mvkpf', 'crowd-counting', 1),
    ('object-detection-9rg5e', 'crowd-detection', 1),
    ('crowdcounting-6lxcc', 'crowd-counting-qqecb', 1),
    ('people-detection-sqprc', 'people-detection-general', 1),
]

for ws, proj, ver in crowd_datasets:
    try:
        project = rf.workspace(ws).project(proj)
        dataset_crowd = project.version(ver).download('yolov8')
        print(f'\u2705 Crowd dataset: {ws}/{proj} v{ver} \u2014 {dataset_crowd.location}')
        break
    except:
        print(f'\u2014 {ws}/{proj} v{ver} not available')
        continue

# Fallback: use primary dataset with crowd-focused augmentation
if dataset_crowd is None:
    print('\u26a0\ufe0f No crowd dataset found \u2014 falling back to primary dataset')
    try:
        project = rf.workspace('footballplayertracking').project('jerseynumberdetectordigitdetector')
        dataset_crowd = project.version(1).download('yolov8')
        print(f'\u2705 Fallback: {dataset_crowd.location}')
    except:
        print('\u274c Primary dataset also failed')
        dataset_crowd = None

In [ ]:
if dataset_crowd is not None:
    print('=' * 60)
    print('TRAINING: crowd_energy_detector_v4 (epochs=100)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_crowd.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='crowd_energy_detector_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.15, device=0,
    )
    print('\u2705 Training complete: crowd_energy_detector_v4')
else:
    print('\u23ed\ufe0f SKIPPED: crowd_energy_detector_v4 \u2014 dataset not available')

In [ ]:
download_model('crowd_energy_detector_v4.pt', 'crowd_energy_detector_v4')

# ================================================================
# SECTION E: Niche Scenario Specialists (Models 15-20)
# ================================================================
#
# These models handle specific challenging conditions:
# - Night games (harsh stadium lights)
# - Indoor courts (gym lighting, floor reflections)
# - Crowd obstruction (spectators blocking view)
# - Helmet glare (sun glare on outdoor football)
# - Low resolution (older/phone game film)
# - Multi-player clusters (pile-ups, post play, crease)
#
# Models 15, 17-19 reuse the primary dataset with extreme augmentation.
# Model 16 reuses basketball dataset. Model 20 merges multiple datasets.
#
# 6 models, ~3 hours on A100

In [ ]:
# \u2500\u2500 Re-download primary dataset if needed \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# Models 15, 17, 18, 19 all reuse this dataset
if 'dataset_primary' not in dir() or dataset_primary is None:
    print('Re-downloading primary dataset for specialist training...')
    try:
        project = rf.workspace('footballplayertracking').project('jerseynumberdetectordigitdetector')
        dataset_primary = project.version(1).download('yolov8')
        print(f'\u2705 Downloaded: {dataset_primary.location}')
    except Exception as e:
        print(f'v1 failed: {e}')
        try:
            dataset_primary = project.version(0).download('yolov8')
            print(f'\u2705 Downloaded v0: {dataset_primary.location}')
        except:
            print(f'\u274c Both versions failed')
            dataset_primary = None
else:
    print(f'\u2705 Primary dataset already loaded: {dataset_primary.location}')

if dataset_primary is None:
    print('\u274c CRITICAL: Primary dataset unavailable \u2014 specialists 15, 17-19 cannot train.')
else:
    print('\u2705 Primary dataset ready for specialist training')

### Model 15: night_game_specialist_v4.pt
Night game specialist. Stadium lighting creates harsh conditions — this model handles them.

**Dataset:** Primary (13,815 images) with extreme dark augmentation  
**Tier:** epochs=150, extreme HSV (hsv_v=0.9)

In [ ]:
if dataset_primary is not None:
    print('=' * 60)
    print('TRAINING: night_game_specialist_v4 (EXTREME dark augmentation)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_primary.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='night_game_specialist_v4',
        augment=True, hsv_h=0.03, hsv_s=0.9, hsv_v=0.9,
        degrees=25, translate=0.2, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.2, copy_paste=0.2,
        erasing=0.4, device=0,
    )
    print('\u2705 Training complete: night_game_specialist_v4')
else:
    print('\u23ed\ufe0f SKIPPED: night_game_specialist_v4 \u2014 primary dataset not available')

In [ ]:
download_model('night_game_specialist_v4.pt', 'night_game_specialist_v4')

### Model 16: indoor_court_specialist_v4.pt
Indoor court specialist. Handles gym lighting, floor reflections, and close angle cameras.

**Dataset:** computer-vision-d5fjh/basketball-detection-dn6fg v1 with indoor-specific augmentation  
**Tier:** epochs=100

In [ ]:
# Reuse basketball hoop dataset for indoor court specialist
if 'dataset_hoop' not in dir() or dataset_hoop is None:
    try:
        project = rf.workspace('computer-vision-d5fjh').project('basketball-detection-dn6fg')
        dataset_hoop = project.version(1).download('yolov8')
        print(f'\u2705 {dataset_hoop.location}')
    except:
        print('\u274c Failed to download basketball dataset')
        dataset_hoop = None
else:
    print(f'\u2705 Reusing: {dataset_hoop.location}')

In [ ]:
if dataset_hoop is not None:
    print('=' * 60)
    print('TRAINING: indoor_court_specialist_v4 (indoor-specific augmentation)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_hoop.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='indoor_court_specialist_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.15, device=0,
    )
    print('\u2705 Training complete: indoor_court_specialist_v4')
else:
    print('\u23ed\ufe0f SKIPPED: indoor_court_specialist_v4 \u2014 dataset not available')

In [ ]:
download_model('indoor_court_specialist_v4.pt', 'indoor_court_specialist_v4')

### Model 17: crowd_obstruction_specialist_v4.pt
Crowd obstruction handler. High school games often have spectators close to sidelines blocking view.

**Dataset:** Primary (13,815 images) with extreme copy_paste=0.6 and erasing=0.7  
**Tier:** epochs=150, extreme occlusion augmentation

In [ ]:
if dataset_primary is not None:
    print('=' * 60)
    print('TRAINING: crowd_obstruction_specialist_v4 (EXTREME occlusion)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_primary.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='crowd_obstruction_specialist_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.6,
        degrees=25, translate=0.2, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.3, copy_paste=0.6,
        erasing=0.7, device=0,
    )
    print('\u2705 Training complete: crowd_obstruction_specialist_v4')
else:
    print('\u23ed\ufe0f SKIPPED: crowd_obstruction_specialist_v4 \u2014 primary dataset not available')

In [ ]:
download_model('crowd_obstruction_specialist_v4.pt', 'crowd_obstruction_specialist_v4')

### Model 18: helmet_glare_specialist_v4.pt
Helmet glare specialist. Sun glare on helmets is a major challenge for jersey number reading at outdoor day games.

**Dataset:** Primary (13,815 images) with extreme hsv_v + brightness variation  
**Tier:** epochs=150

In [ ]:
if dataset_primary is not None:
    print('=' * 60)
    print('TRAINING: helmet_glare_specialist_v4 (EXTREME brightness/glare)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_primary.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='helmet_glare_specialist_v4',
        augment=True, hsv_h=0.03, hsv_s=0.9, hsv_v=0.9,
        degrees=25, translate=0.2, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.2, copy_paste=0.2,
        erasing=0.4, device=0,
    )
    print('\u2705 Training complete: helmet_glare_specialist_v4')
else:
    print('\u23ed\ufe0f SKIPPED: helmet_glare_specialist_v4 \u2014 primary dataset not available')

In [ ]:
download_model('helmet_glare_specialist_v4.pt', 'helmet_glare_specialist_v4')

### Model 19: low_resolution_specialist_v4.pt
Low resolution specialist. Handles older game film recorded on basic cameras or phones.

**Dataset:** Primary (13,815 images) with heavy blur + downscale simulation  
**Tier:** epochs=150, blur=0.7

In [ ]:
if dataset_primary is not None:
    print('=' * 60)
    print('TRAINING: low_resolution_specialist_v4 (EXTREME blur/downscale)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_primary.location}/data.yaml',
        epochs=150, imgsz=832, batch=8,
        name='low_resolution_specialist_v4',
        augment=True, hsv_h=0.03, hsv_s=0.9, hsv_v=0.7,
        degrees=30, translate=0.3, scale=0.8, fliplr=0.5,
        mosaic=1.0, mixup=0.3, copy_paste=0.3,
        erasing=0.5, device=0,
    )
    print('\u2705 Training complete: low_resolution_specialist_v4')
else:
    print('\u23ed\ufe0f SKIPPED: low_resolution_specialist_v4 \u2014 primary dataset not available')

In [ ]:
download_model('low_resolution_specialist_v4.pt', 'low_resolution_specialist_v4')

### Model 20: multi_player_cluster_v4.pt
Multi-player cluster detection. Identifies target player even when surrounded by defenders. Critical for post play, pile-ups, crease battles.

**Dataset:** MERGED — football + basketball + lacrosse player detection  
**Tier:** epochs=100, high mosaic=1.0, copy_paste=0.5

In [ ]:
# Use the best available player detection dataset
dataset_cluster = None

# Priority: primary (largest), then basketball, then lacrosse
if 'dataset_primary' in dir() and dataset_primary is not None:
    dataset_cluster = dataset_primary
    print(f'\u2705 Using primary: {dataset_cluster.location}')
elif 'dataset_hoop' in dir() and dataset_hoop is not None:
    dataset_cluster = dataset_hoop
    print(f'\u2705 Using basketball: {dataset_cluster.location}')
elif 'dataset_lacrosse' in dir() and dataset_lacrosse is not None:
    dataset_cluster = dataset_lacrosse
    print(f'\u2705 Using lacrosse: {dataset_cluster.location}')
else:
    # Re-download primary
    try:
        project = rf.workspace('footballplayertracking').project('jerseynumberdetectordigitdetector')
        dataset_cluster = project.version(1).download('yolov8')
        print(f'\u2705 Downloaded: {dataset_cluster.location}')
    except:
        print('\u274c No dataset available for multi-player cluster')
        dataset_cluster = None

In [ ]:
if dataset_cluster is not None:
    print('=' * 60)
    print('TRAINING: multi_player_cluster_v4 (high mosaic + copy_paste)')
    print('=' * 60)
    model = YOLO(V4_BASE)
    model.train(
        data=f'{dataset_cluster.location}/data.yaml',
        epochs=100, imgsz=832, batch=8,
        name='multi_player_cluster_v4',
        augment=True, hsv_h=0.02, hsv_s=0.9, hsv_v=0.5,
        degrees=20, translate=0.2, scale=0.7, fliplr=0.5,
        mosaic=1.0, mixup=0.15, copy_paste=0.5, device=0,
    )
    print('\u2705 Training complete: multi_player_cluster_v4')
else:
    print('\u23ed\ufe0f SKIPPED: multi_player_cluster_v4 \u2014 dataset not available')

In [ ]:
download_model('multi_player_cluster_v4.pt', 'multi_player_cluster_v4')

# ================================================================
# FINAL SUMMARY
# ================================================================

In [ ]:
import os

print('=' * 70)
print('v4 OUTCOME DETECTION TRAINING COMPLETE \u2014 FINAL REPORT')
print('=' * 70)

all_v4_models = [
    ('basketball_hoop_detector_v4.pt',       'Section A \u2014 Basketball Outcome',   'Detects hoop position for made shot confirmation'),
    ('basketball_made_shot_v4.pt',           'Section A \u2014 Basketball Outcome',   'Ball + net detection for scoring'),
    ('basketball_scoring_zone_v4.pt',        'Section A \u2014 Basketball Outcome',   'Court zones: 2pt vs 3pt context'),
    ('basketball_dribble_drive_v4.pt',       'Section A \u2014 Basketball Outcome',   'Ball handling / drive detection'),
    ('basketball_rebound_v4.pt',             'Section A \u2014 Basketball Outcome',   'Rebound detection'),
    ('football_completion_detector_v4.pt',   'Section B \u2014 Football Outcome',    'Pass completion vs incomplete'),
    ('football_touchdown_detector_v4.pt',    'Section B \u2014 Football Outcome',    'Touchdown detection'),
    ('football_sack_detector_v4.pt',         'Section B \u2014 Football Outcome',    'Sack detection (defensive)'),
    ('football_reception_yac_v4.pt',         'Section B \u2014 Football Outcome',    'Reception + yards after catch'),
    ('football_qb_scramble_v4.pt',           'Section B \u2014 Football Outcome',    'QB scramble (Dustin #11)'),
    ('lacrosse_goal_detector_v4.pt',         'Section C \u2014 Lacrosse Outcome',    'Goal detection'),
    ('lacrosse_shot_quality_v4.pt',          'Section C \u2014 Lacrosse Outcome',    'Shot quality context'),
    ('lacrosse_ground_ball_v4.pt',           'Section C \u2014 Lacrosse Outcome',    'Ground ball detection'),
    ('crowd_energy_detector_v4.pt',          'Section D \u2014 Cross-Sport',         'Crowd reaction validator'),
    ('night_game_specialist_v4.pt',          'Section E \u2014 Niche Specialist',    'Night game / stadium lights'),
    ('indoor_court_specialist_v4.pt',        'Section E \u2014 Niche Specialist',    'Indoor gym lighting'),
    ('crowd_obstruction_specialist_v4.pt',   'Section E \u2014 Niche Specialist',    'Crowd obstruction handler'),
    ('helmet_glare_specialist_v4.pt',        'Section E \u2014 Niche Specialist',    'Sun/helmet glare'),
    ('low_resolution_specialist_v4.pt',      'Section E \u2014 Niche Specialist',    'Low res / phone footage'),
    ('multi_player_cluster_v4.pt',           'Section E \u2014 Niche Specialist',    'Multi-player pile-ups'),
]

passed = []
failed = []
for name, section, purpose in all_v4_models:
    if os.path.exists(name):
        size_mb = os.path.getsize(name) / 1024 / 1024
        try:
            from ultralytics import YOLO
            m = YOLO(name)
            metrics = m.val()
            map50 = metrics.box.map50
            status = '\u2705 PASS' if map50 >= 0.4 else '\u26a0\ufe0f LOW'
            passed.append(f'  {status}: {name} ({size_mb:.1f}MB, mAP50={map50:.3f}) \u2014 {section}')
        except Exception:
            passed.append(f'  \u2705 DOWNLOADED: {name} ({size_mb:.1f}MB) \u2014 {section}')
    else:
        run_path = f'runs/detect/{name.replace(".pt", "")}/weights/best.pt'
        if os.path.exists(run_path):
            failed.append(f'  \u26a0\ufe0f TRAINED but NOT DOWNLOADED: {name} \u2014 run its download cell! \u2014 {section}')
        else:
            failed.append(f'  \u274c MISSING: {name} \u2014 not trained \u2014 {section}')

print(f'\nDOWNLOADED ({len(passed)}/20):')
for m in passed:
    print(m)

if failed:
    print(f'\nNOT DOWNLOADED ({len(failed)}/20):')
    for m in failed:
        print(m)
else:
    print('\n\ud83c\udf89 ALL 20 MODELS DOWNLOADED!')

print()
print('\u2500' * 70)
print('FILENAME CROSS-CHECK (must match roboflow_detector.py + health.py):')
print('\u2500' * 70)
expected = [
    'basketball_hoop_detector_v4.pt',
    'basketball_made_shot_v4.pt',
    'basketball_scoring_zone_v4.pt',
    'basketball_dribble_drive_v4.pt',
    'basketball_rebound_v4.pt',
    'football_completion_detector_v4.pt',
    'football_touchdown_detector_v4.pt',
    'football_sack_detector_v4.pt',
    'football_reception_yac_v4.pt',
    'football_qb_scramble_v4.pt',
    'lacrosse_goal_detector_v4.pt',
    'lacrosse_shot_quality_v4.pt',
    'lacrosse_ground_ball_v4.pt',
    'crowd_energy_detector_v4.pt',
    'night_game_specialist_v4.pt',
    'indoor_court_specialist_v4.pt',
    'crowd_obstruction_specialist_v4.pt',
    'helmet_glare_specialist_v4.pt',
    'low_resolution_specialist_v4.pt',
    'multi_player_cluster_v4.pt',
]
for name in expected:
    exists = '\u2705' if os.path.exists(name) else '\u274c'
    print(f'  {exists} {name}')

print()
print('SERVICE FILE MAPPING (which file uses each model):')
print('\u2500' * 70)
service_map = {
    'basketball_hoop_detector_v4.pt': 'game_stats.py \u2014 made shot confirmation',
    'basketball_made_shot_v4.pt': 'game_stats.py \u2014 points_estimated, shots_made',
    'basketball_scoring_zone_v4.pt': 'game_stats.py \u2014 2pt vs 3pt zone',
    'basketball_dribble_drive_v4.pt': 'clip_extractor.py \u2014 drive highlight scoring',
    'basketball_rebound_v4.pt': 'game_stats.py \u2014 rebound counting',
    'football_completion_detector_v4.pt': 'game_stats.py \u2014 completion_percentage',
    'football_touchdown_detector_v4.pt': 'game_stats.py + clip_extractor.py \u2014 TD = Elite grade',
    'football_sack_detector_v4.pt': 'game_stats.py \u2014 sack counting',
    'football_reception_yac_v4.pt': 'clip_extractor.py \u2014 YAC scoring boost',
    'football_qb_scramble_v4.pt': 'clip_extractor.py \u2014 QB scramble highlight',
    'lacrosse_goal_detector_v4.pt': 'game_stats.py + clip_extractor.py \u2014 goal = Elite grade',
    'lacrosse_shot_quality_v4.pt': 'clip_extractor.py \u2014 shot quality scoring',
    'lacrosse_ground_ball_v4.pt': 'game_stats.py \u2014 ground_balls counting',
    'crowd_energy_detector_v4.pt': 'clip_extractor.py \u2014 big play validator (all sports)',
    'night_game_specialist_v4.pt': 'roboflow_detector.py \u2014 night game OCR accuracy',
    'indoor_court_specialist_v4.pt': 'roboflow_detector.py \u2014 indoor OCR accuracy',
    'crowd_obstruction_specialist_v4.pt': 'roboflow_detector.py \u2014 occlusion handling',
    'helmet_glare_specialist_v4.pt': 'roboflow_detector.py \u2014 glare handling',
    'low_resolution_specialist_v4.pt': 'roboflow_detector.py \u2014 low res handling',
    'multi_player_cluster_v4.pt': 'roboflow_detector.py \u2014 cluster detection',
}
for name, usage in service_map.items():
    print(f'  {name} \u2192 {usage}')

print()
print('GIT COMMANDS (run after moving .pt files to app/model/):')
print('  cd playerJerseyIdentification-master')
print('  cp *.pt app/model/')
print('  git add app/model/*.pt')
print('  git commit -m "Add v4 outcome detection models \u2014 20 play outcome + niche specialists"')
print('  git push')

print()
print('=' * 70)
print(f'v4 OUTCOME DETECTION: {len(passed)}/20 models ready')
print('=' * 70)

In [ ]:
print('''
V4 INTEGRATION NOTES:
These models add OUTCOME CONTEXT to existing detections.
They don't replace v1/v2/v3 \u2014 they add a layer on top.

When integrated:
- Clip grade boosted if outcome detector confirms success
- basketball_made_shot + basketball_hoop = +25 to clip score
- football_touchdown = clip auto-graded as Elite
- football_completion = +15 to clip score
- lacrosse_goal = clip auto-graded as Elite
- Specialist models (night/indoor/glare) improve OCR accuracy
  in specific scenarios
- crowd_energy_detector validates any big play across sports

SCORING TIERS (clip_extractor.py):
- Elite (>=90): any confirmed outcome (made shot, TD, goal)
- Strong (70-89): highlight play without confirmed outcome
- Decent (50-69): player involved but unclear outcome
- Cut (<50): player not involved or dead ball

FULL DETECTION LAYER ORDER (after v4 deployed):
Step 1: Ali's ensemble (v1) runs first
Step 2: jersey_number_universal_v1 (v2, mAP50 0.995)
Step 3: v3 OCR pipeline (12 models)
Step 4: v2 sport-specific models
Step 5: temporal_consensus (3+ frame filter)
Step 6: v4 outcome detection (these 20 models)
Step 7: Stat generation (zones, actions, ball tracking)
''')

# ================================================================
# AFTER TRAINING — NEXT STEPS
# ================================================================
#
# 1. Move all downloaded .pt files to jersey-detection/app/model/
# 2. Run: git add app/model/*.pt
#         git commit -m "Add v4 outcome detection models"
#         git push
# 3. Railway auto-deploys in ~3 minutes
# 4. Verify with health check:
#    curl https://jersey-detection-production-d8d8.up.railway.app/health
#    Look for roboflow_models_v4 — all should show 'loaded'
# 5. Run the primary test (Dustin's game film):
#    curl -X POST .../analyze \
#      -H 'Content-Type: application/json' \
#      -d '{"videoUrl":"https://www.youtube.com/watch?v=XRSrRPbZIF0",
#           "jerseyNumber":11,"jerseyColor":"navy","sport":"football",
#           "position":"QB","timeRangeStart":60,"timeRangeEnd":120}'
#    SUCCESS = detections > 0, outcome models contributed
#
# v4 adds OUTCOME CONTEXT on top of v1/v2/v3.
# It does NOT replace any existing models.